In [ ]:
import pandas as pd
import torch as pt
pt.set_default_dtype(pt.float64)

import os
import dill

import sys
PATH_CNMc = '../../../'
sys.path.append(PATH_CNMc)

# reproducibility
pt.manual_seed(4224)

# plot Settings
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'serif' # or 'sans-serif' or 'monospace'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams['font.sans-serif'] = 'cmss10'
plt.rcParams['font.monospace'] = 'cmtt10'
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["axes.formatter.use_mathtext"] = True # fixes the minus sign


**Download data**

In [ ]:
# Download and unpack Lorenz data
os.makedirs('./data/', exist_ok=True)
!curl -L https://zenodo.org/records/19450335/files/Lorenz63.tar.xz -o ./data/Lorenz63.tar.xz
!cd ./data; tar -xvf Lorenz63.tar.xz

**Config datasets**

In [ ]:
# data config
DATAPATH = './data/'
casepath = './output/'

FILENAMES = [
            'Lorenz_b30.000.h5',
            'Lorenz_b40.000.h5',
            'Lorenz_b50.000.h5',
            'Lorenz_b60.000.h5',
            'Lorenz_b70.000.h5',
             ]
PARAMETERS = [(30.0,), (40.0,), (50.0,), (60.0,), (70.0,)]

# train/test OC
ocs_train = [(30.0,), (40.0,), (60.0,), (70.0,)]
ocs_test = [(50.0,),]

# train/test trajectory split
ti_train = -400_000
ti_test = -800_000


**Read data**

In [ ]:
data={}
test_data={}

for filename in FILENAMES:
    d = pd.read_hdf(DATAPATH+filename, key='data', header=None, index_col=0)[[1,2,3,4]].to_numpy()
    oc = (float(os.path.splitext(filename)[0].split('_')[-1][1:]),)
    dt = d[1,0]-d[0,0]
    d = pt.from_numpy(d.T)
    print(d.shape)
    # train/test splits:
    data[oc] = d[1:4, ti_train:]
    test_data[oc] = d[1:4, ti_test:ti_train]

T = dt * data[oc].shape[-1] # total trajectory time

# training OCs
data_train = {oc: data[oc] for oc in ocs_train}
# test OCs
data_test = {oc: data[oc] for oc in ocs_test}


In [ ]:
# Visualise data

# %matplotlib ipympl
%matplotlib inline

fig = plt.figure(figsize=(4,4), dpi=160)
ax = fig.add_subplot(1,1,1, projection='3d', proj_type='ortho',computed_zorder=False)
ax.view_init(15, 80)

for oc, traj_data in data.items():
    coarse = traj_data[:,:100_000:1]
    x = coarse[0,:]
    y = coarse[1,:]
    z = coarse[2,:]
    # ax.scatter(x, y, z, alpha=0.4, s=1, zorder=1, label=PARAMETERS[i])
    ax.plot(x, y, z, alpha=0.6, zorder=1, lw=0.1, label=oc[0])

ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.set_xlabel('$x$'); ax.set_ylabel('$y$'); ax.set_zlabel('$z$')
plt.legend(frameon=False, bbox_to_anchor=(1.0, 0.7))
ax.set_axis_off()
plt.tight_layout()
# plt.savefig(casepath+'datavis.png')
plt.show()

**Set up CNMc components**

In [ ]:
# cnm/cnmc set up
K=14    # no. clusters
L=1     # no. past delays (markov order)

outpath=casepath+'K{:d}_L{:d}/'.format(K,L)
os.makedirs(outpath, exist_ok=True)

# initialise Procrustes transformations
from cnmc.encoders.ProcrustesEncoder import ProcrustesEncoder
oc_reference = ocs_train[0]  # reference OC other OCs are mapped to
encoders = [ProcrustesEncoder(reference_state=data_train[oc_reference]) for oc in ocs_train]
# pick supervised transformation model
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from cnmc.encoders.procrustes_model import procrustes_model
encoder_base = make_pipeline(MinMaxScaler(), LinearRegression())
encoder_model = procrustes_model(base_model=encoder_base, 
                                 )

# set up clustering
from sklearn.cluster import KMeans
from cnmc.cnmc.utils_train import read_clusters_from_file
compute_clusters = True        # compute clusters? False=read from csv file
cluster_config = {'n_clusters': K, 'n_init': 10, 'max_iter': 1_000, 'init': 'k-means++'}
clustering = {'algorithm': KMeans(**cluster_config)}
clusters_csv = outpath+'clusters.csv'
if not compute_clusters: # optionally read centroids from file
    clusters_from_file = read_clusters_from_file(clusters_csv, **cluster_config)
    clustering = {'algorithm': clusters_from_file}

# set up and init CNM models and store into ROMList 
from cnmc.crom import CNM
from cnmc import ROMList
cnm_config = {'dt': dt, 
              'encoder': None, 
              'n_clusters': K, 
              'model_order': L,
              'spline_order': 1,}
cnms = ROMList([{"oc": oc, "rom": CNM(**cnm_config)} for oc in ocs_train])

# pick transition properties model (\hat{Q}(\alpha), \hat{T}(\alpha))
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler
from sklearn.linear_model import Lasso
transition_model = make_pipeline(MinMaxScaler(), PolynomialFeatures(degree=3), Lasso(1e-2))


**Build and train CNMc**

In [ ]:
# initialise CNMc
from cnmc import CNMc
cnmc = CNMc(cnms, 
            encoders=encoders,
            clustering=clustering, 
            encoder_model=procrustes_model(),
            transition_model=transition_model
            )

# train CNMc
cnmc.train(data_train)

# writing out models
with open(outpath+'model_cnmc.pkl', 'wb') as f:
    dill.dump(cnmc, f)

**Train reference CNM at test condition**

In [ ]:
# train the reference test CNM
from cnmc.cnmc.utils_train import train_croms, train_clusters

cnms_test = ROMList([{"oc": oc, "rom": CNM(**cnm_config)} for oc in ocs_test])
encoders_test = {oc: ProcrustesEncoder(reference_state=data_train[oc_reference]).train(data_test[oc]) for oc in ocs_test}

# set up clustering
compute_ocs_clusters = True     # compute test clusters?
cluster_config = {'n_clusters': K, 'n_init': 10, 'max_iter': 1_000, 'init': 'k-means++'}
clusters_ocs_csv = outpath+'clusters_test.csv'
if compute_ocs_clusters:
    clustering_test = {'algorithm': KMeans(**cluster_config)}
    data_test_encoded = {oc: encoders_test[oc].encode(data_test[oc]) for oc in ocs_test}
    clustering_test = train_clusters(clustering_test, data_test_encoded, output_csv=clusters_ocs_csv)
    del data_test_encoded
else:
    clusters_from_file = read_clusters_from_file(clusters_ocs_csv, **cluster_config)
    clustering_test = {'algorithm': clusters_from_file}

cnmlist_test = train_croms(cnms_test.roms, data_test, encoders_test, clustering_test, encode=True)


models_cnm_test = {param: trained_cnm for param, trained_cnm in zip(ocs_test, cnmlist_test)}
with open(outpath+'models_cnm_test.pkl', 'wb') as f:
    dill.dump(models_cnm_test, f)

**Evaluate models at unseen OC**

In [ ]:
# Dynamics generation

# Test Predictions
print('Testing at: ', ocs_test)

ocs_test_pt=pt.Tensor(ocs_test)      # batch of 1 OCs
print('ocs_test_pt', ocs_test_pt)
cnms_oc = cnmc.predict_model(ocs_test_pt) # predict CROM model at unseen OC

# CNM true trajectory.
true_traj = test_data[ocs_test[0]]
print(true_traj.shape)

# initial state
x0 = pt.Tensor(true_traj[:,1_000]).unsqueeze(-1) 

# generate CNM reference 
test_traj_cnm = cnmlist_test[0].predict(x0,T,dt)
pd.DataFrame(test_traj_cnm.T).to_csv(outpath+'traj_cnm.csv', header=None)

# generate CNMc trajectory
test_traj_cnmc = cnmc.predict(ocs_test_pt, x0, T,dt)[ocs_test[0]]
pd.DataFrame(test_traj_cnmc.T).to_csv(outpath+'traj_cnmc.csv', header=None)


In [ ]:
# Plot trajectory
xa, xb = 2000, 4000
plt.figure(figsize=(8,1.5), dpi=200)
plt.plot(true_traj[1,xa:xb], c='k', alpha=0.5, label='numerical', lw=3)
crd = 1
plt.plot(test_traj_cnm[crd,xa:xb], c='b', label='cnm')
plt.plot(test_traj_cnmc[crd,xa:xb], c='r', label='cnmc')
# plt.legend(frameon=False,)
plt.xlim((0, xb-xa))
plt.xlabel('$t$')
plt.ylabel(f'$u_{crd+1}$')
# plt.title(r'{\sf\textbf{A}}', loc='left')
plt.tight_layout()
plt.savefig(outpath+'time_series.png')
plt.show()